# Session 1: a narrowing pipe speeds the water up and lowers its pressure
**A. Real-world problem.** Put your thumb over the end of a garden hose and the water
shoots out faster. Water cannot pile up inside the pipe, so where the pipe is narrower the
water must move faster. And a faster stream has used some of its pressure to get fast.

**Plan.** (1) continuity: how much faster?, (2) Bernoulli: how much pressure is traded?,
(3) a pipe that narrows in the middle: speed and pressure along its length, (4) plot it.
This is an idealised model: no friction (that comes in Session 2).

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

RHO = 998.0      # density of water at 20 C, kg/m^3

## Step 1: continuity (water is not squeezed or lost)
The volume flow rate Q (cubic metres per second) is the same at every cross-section of a
steady pipe. A circle of diameter D has area A = pi D^2 / 4, so Q = A v and

    A1 v1 = A2 v2     ->     v2 = v1 * (D1 / D2)^2

Halve the diameter and the area drops to a quarter, so the speed rises four times.

In [ ]:
def area(d):
    """Cross-section area (m^2) of a round pipe of diameter d (m)."""
    return np.pi * np.asarray(d, dtype=float) ** 2 / 4.0


def velocity(q, d):
    """Average speed (m/s) for flow rate q (m^3/s) in a pipe of diameter d (m)."""
    return q / area(d)

## Step 2: Bernoulli (speed is paid for with pressure)
For steady, incompressible, frictionless flow along a level pipe, the sum
p + (1/2) rho v^2 stays constant. The second term is the *dynamic pressure*. If v goes up,
p must go down by the same amount:

    p2 = p1 + (1/2) rho (v1^2 - v2^2)

In [ ]:
def pressure_after(p1, v1, v2, rho=RHO):
    """Static pressure (Pa) after the speed changes from v1 to v2 (m/s), level pipe, no friction."""
    return p1 + 0.5 * rho * (v1 ** 2 - v2 ** 2)


def total_pressure(p, v, rho=RHO):
    """p + 1/2 rho v^2 (Pa): constant along the pipe in the ideal model."""
    return p + 0.5 * rho * v ** 2

## Step 3: a pipe that narrows in the middle
The pipe has diameter D1 on the left, D2 on the right, and a smooth taper in between
(a smoothstep curve from 40% to 60% of the length). At every x we apply continuity and then
Bernoulli. Pressure is *gauge* pressure: measured above the outside air.

In [ ]:
def diameter_profile(x, d1, d2, length=2.0):
    """Diameter (m) at distance x (m) along a pipe of the given length, narrowing smoothly."""
    t = np.clip((np.asarray(x, dtype=float) / length - 0.4) / 0.2, 0.0, 1.0)
    s = t * t * (3 - 2 * t)
    return d1 + (d2 - d1) * s


def profile(q, d1, d2, p1, length=2.0, n=201):
    """Return x, diameter, speed and gauge pressure along the pipe."""
    x = np.linspace(0.0, length, n)
    d = diameter_profile(x, d1, d2, length)
    v = velocity(q, d)
    p = pressure_after(p1, v[0], v)
    return x, d, v, p

## Step 4: see it
Left: pipe shape, speed and pressure share the same x axis. Right: how the speed gain
depends on the diameter ratio D1/D2 (it is the square of the ratio).

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    q, d1, d2, p1 = 0.010, 0.100, 0.050, 100e3
    x, d, v, p = profile(q, d1, d2, p1)
    print(f"v1 = {v[0]:.3f} m/s, v2 = {v[-1]:.3f} m/s, p2 = {p[-1] / 1e3:.2f} kPa "
          f"(drop {(p[0] - p[-1]) / 1e3:.2f} kPa)")
    print(f"total pressure start {total_pressure(p[0], v[0]) / 1e3:.2f} kPa, "
          f"end {total_pressure(p[-1], v[-1]) / 1e3:.2f} kPa")

    fig, ax = plt.subplots(3, 1, figsize=(6, 6.5), sharex=True)
    ax[0].fill_between(x, -d / 2 * 1000, d / 2 * 1000, color="tab:blue", alpha=0.35)
    ax[0].set_ylabel("pipe radius (mm)")
    ax[0].set_title("Narrowing pipe: Q = 10 L/s, p1 = 100 kPa (gauge)")
    ax[1].plot(x, v, color="tab:green")
    ax[1].set_ylabel("speed (m/s)")
    ax[2].plot(x, p / 1e3, color="tab:red")
    ax[2].set_ylabel("gauge pressure (kPa)")
    ax[2].set_xlabel("distance along pipe (m)")
    for a in ax:
        a.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "narrowing_pipe.png", dpi=120)

    ratio = np.linspace(1, 3, 101)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(ratio, ratio ** 2)
    ax2.set_xlabel("diameter ratio D1 / D2 (no unit)")
    ax2.set_ylabel("speed ratio v2 / v1 (no unit)")
    ax2.set_title("Speed gain is the square of the diameter ratio")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "speed_gain.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the diameters in `lesson.py` to 80 mm and 40 mm and the flow to 5 L/s. Predict v2 and p2 first, then compare with Section F question B.
2. **Stretch.** Write `throat_diameter(q, vmax)` that returns the smallest diameter (m) for which the speed does not exceed vmax, and test it with q = 0.02 m^3/s and vmax = 6 m/s.
3. **Explain.** Explain, with the idea of energy, why the total p + (1/2) rho v^2 would fall along a long real pipe even though the diameter does not change.

## Assessment
- **A. Conceptual.** A pipe narrows. Explain in words why the water speeds up and why its static pressure falls, and name one thing the idealised model leaves out.
- **B. Numerical.** Water at 5 L/s flows at 150 kPa gauge through an 80 mm pipe that narrows to 40 mm. Find v1, v2 and p2. Check the units.
- **C. Graph and simulation interpretation.** In the app, set the narrow diameter larger than the wide one (a pipe that widens). What does the pressure graph do, and is this realistic?
- **D. Design and optimisation challenge.** Design: Q = 20 L/s flows at 100 kPa gauge in a 100 mm pipe. A flow-measuring throat must reach a speed of at least 5 m/s, but the gauge pressure must stay above 80 kPa. Choose a throat diameter from 50, 60, 70, 80 mm and justify it.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    v1 = L.velocity(0.010, 0.100)
    v2 = L.velocity(0.010, 0.050)
    checks.append(("worked example: 10 L/s gives 1.273 m/s in 100 mm and 5.093 m/s in 50 mm",
                   abs(v1 - 1.2732) < 1e-3 and abs(v2 - 5.0930) < 1e-3))
    p2 = L.pressure_after(100e3, v1, v2)
    checks.append((f"worked example: pressure after the narrowing is 87.87 kPa (got {p2 / 1e3:.2f})",
                   abs(p2 / 1e3 - 87.87) < 0.01))
    checks.append(("hand calculation: halving the diameter quadruples the speed", abs(v2 / v1 - 4.0) < 1e-9))
    checks.append(("continuity: A1 v1 = A2 v2 = Q",
                   abs(L.area(0.100) * v1 - 0.010) < 1e-12 and abs(L.area(0.050) * v2 - 0.010) < 1e-12))
    x, d, v, p = L.profile(0.010, 0.100, 0.050, 100e3)
    tot = L.total_pressure(p, v)
    checks.append(("Bernoulli: p + 1/2 rho v^2 is constant along the pipe", np.allclose(tot, tot[0], atol=1e-6)))
    checks.append(("pressure falls where the pipe is narrow", p[-1] < p[0] and np.all(np.diff(p) <= 1e-9)))
    x, d, v, p = L.profile(0.010, 0.080, 0.080, 50e3)
    checks.append(("limit: equal diameters change neither speed nor pressure",
                   np.allclose(v, v[0]) and np.allclose(p, 50e3)))
    x, d, v, p = L.profile(0.010, 0.050, 0.100, 100e3)
    checks.append(("a widening pipe recovers pressure (p rises above the start)", p[-1] > p[0]))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
